# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We demonstrate step-by-step how to load the Croissant schema, inspect record sets, reference all entities by their `@id` fields, and perform initial data exploration.

### Dataset Source
The dataset is accessible as a Croissant JSON-LD schema at:
[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript, access as object attributes)
print(f"Dataset loaded: {dataset.metadata.name}\n{dataset.metadata.description}")

# Display basic attributes
print(f"- Identifier: {dataset.metadata.identifier}")
print(f"- License: {dataset.metadata.license}")
print(f"- Version: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, fields, and their `@id` values.

**Note:** In Croissant, each entity has a unique `@id`, which is used to unambiguously refer to record sets, fields, and columns.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record sets in the dataset:\n")
for rs in record_sets:
    print(f"- Record set name: {rs.name}, @id: {rs.id}")

# For each record set, print their fields and columns by @id
for rs in record_sets:
    print(f"\nRecord set: {rs.name}  (@id={rs.id})")
    print(" Fields:")
    for field in rs.fields:
        print(f"   - {field.name} (@id={field.id}, dataType={field.data_type})")
    if hasattr(rs, 'columns') and rs.columns:
        print(" Columns:")
        for column in rs.columns:
            print(f"   - {column.name} (@id={column.id}, dataType={column.data_type})")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for further analysis. All `@id` references are used directly from the schema inspection step above.

Below, a list of record set `@id` is set for loading.

In [ ]:
# Prepare to load records as DataFrames using @id for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # Load records, referencing record set by @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set @id: {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample records:\n{df.head(2)}\n")

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalization, and grouping using `@id` field references. For demonstration, we select one record set and one numeric field.

In [ ]:
# Select a record set and a numeric field for analysis
record_set_id = record_set_ids[0]  # You may update this based on printed record set @ids
df = dataframes[record_set_id]

# Inspect all columns to pick a numeric field -- here we try to auto-detect
numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_candidates:
    # Try common names
    for name in [
        'Molecular Weight', 'MolecularWeight', 'mw', 'Abundance', 'abundance', 'CoveragePercent', 'PeptideCount', 'Total Peptides'
    ]:
        if name in df.columns:
            numeric_candidates = [name]
            break
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for demonstration: {numeric_field}")
else:
    print("No obvious numeric field found. Using the first column available.")
    numeric_field = df.columns[0]

# Example filtering and normalization
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
if threshold is None:
    print(f"The field '{numeric_field}' is not numeric and cannot be filtered or normalized.")
else:
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a (possibly string) field if such exists
    group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = None
    possible_names = ['Description', 'description', 'Protein Name', 'Sample', 'sample', 'Type']
    for name in possible_names:
        if name in df.columns:
            group_field = name
            break
    if not group_field and group_candidates:
        group_field = group_candidates[0]

    if group_field:
        grouped_df = filtered_df.groupby(group_field, dropna=False).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize a distribution or relationship for numeric fields in your dataset. Here, a simple histogram and scatter plot are used if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a 2nd numeric field exists, create scatter plot
    if len(numeric_candidates) > 1:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(x=df[numeric_candidates[0]], y=df[numeric_candidates[1]])
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.title(f"{numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.tight_layout()
        plt.show()
else:
    print(f"Cannot plot histogram, field '{numeric_field}' is not numeric.")

## 6. Conclusion
We have loaded, inspected, filtered, and visualized the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, referencing all entities by their unique `@id` as defined in the Croissant schema. Further domain-specific analysis can be performed as needed by referencing fields and record sets by their `@id` for reproducibility and interoperability.